In [1]:
import layoutparser as lp
import os
import pdf2image
from PIL import Image
import pytesseract
import re
import pandas as pd
import sys
sys.path.append('../')
import matplotlib.pyplot as plt
import numpy as np
import pytesseract


In [2]:
model = lp.Detectron2LayoutModel('lp://PrimaLayout/mask_rcnn_R_50_FPN_3x/config')

In [3]:
# function to convert pdf files into images
def pdf_to_img(pdf_file_path, output_images_path,dpi=500,page=None) :

    PDF_PATH = pdf_file_path
    DPI = dpi
    OUTPUT_FOLDER = output_images_path
    FIRST_PAGE = page
    LAST_PAGE = page
    FORMAT = 'jpg'
    THREAD_COUNT = 1
    USERPWD = None
    USE_CROPBOX = False
    STRICT = False

    def rename_filename(output_image_path,idx):
        path, filename = os.path.split(output_image_path)
        os.rename(output_image_path,path+"/"+str(idx)+".jpg")

    def delete_existing_images():
        images = os.listdir(output_images_path)
        for image in images:
            os.remove(output_images_path+"/"+image)



    def pdftopil():

        pil_images = pdf2image.convert_from_path(PDF_PATH,
                                                 dpi=DPI,
                                                 output_folder=OUTPUT_FOLDER,
                                                 first_page=None,
                                                 last_page=None,
                                                 fmt=FORMAT,
                                                 thread_count=THREAD_COUNT,
                                                 userpw=USERPWD,
                                                 use_cropbox=USE_CROPBOX,
                                                 strict=STRICT)

        for idx,image in enumerate(pil_images,1):
            rename_filename(image.filename,idx)

        return pil_images


    delete_existing_images()
    pil_images = pdftopil()

    return pil_images

def crop_section(intial_width,intial_height,crop_width,crop_height,img):
    area = (intial_width, intial_height, intial_width+crop_width, intial_height+crop_height)
    cropped_img = img.crop(area)
    return cropped_img

In [4]:
# convert the pdf into images
pdf_2_images_list = pdf_to_img('TKP_2009_01_08.pdf', 'pdf2images/',dpi=500)

In [5]:
def create_path(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [6]:
columns= ["Page","Block","Text","Image path"]
df = pd.DataFrame(columns=columns)

In [9]:
path = "pdf2images/"
pages = os.listdir(path)

for page in range(1,len(pages)+1):
    
    img = Image.open(path+str(page)+".jpg")

    output_path = "outputs/"+str(page)+"/"
    create_path(output_path)

    layout1 = model.detect(img)

    text_blocks = lp.Layout([b for b in layout1 if b.type=='TextRegion'])
    sep_blocks = lp.Layout([b for b in layout1 if b.type=='SeparatorRegion'])
    figure_blocks = lp.Layout([b for b in layout1 if b.type=='ImageRegion'])

    text_blocks = lp.Layout([b for b in text_blocks \
                   if not any(b.is_in(b_fig) for b_fig in figure_blocks)])

    print("page = ",page)
    print("Length of the layout", len(layout1))
    print("text blocks = ", len(text_blocks))
    print("sep blocks = ", len(sep_blocks))
    print("figure blocks = ", len(figure_blocks))
    print()

    for idx,block in enumerate(text_blocks,1):

        output_file_path = output_path+str(idx)+".png"

        coordinates = block.block
        x1,y1,x2,y2 = coordinates.x_1, coordinates.y_1,coordinates.x_2,coordinates.y_2
        crop_img = crop_section(x1,y1,x2,y2,img)
        crop_img.save(output_file_path)
        text = (pytesseract.image_to_string(output_file_path, config='--psm 6', lang='eng')) #config='--psm 4' config='-c preserve_interword_spaces=1'
        data_list = [page,idx,text,output_file_path]
        df_length = len(df)
        df.loc[df_length] = data_list

page =  1
Length of the layout 92
text blocks =  61
sep blocks =  13
figure blocks =  8

page =  2
Length of the layout 100
text blocks =  83
sep blocks =  7
figure blocks =  3

page =  3
Length of the layout 100
text blocks =  79
sep blocks =  5
figure blocks =  8

page =  4
Length of the layout 79
text blocks =  59
sep blocks =  14
figure blocks =  4

page =  5
Length of the layout 100
text blocks =  86
sep blocks =  5
figure blocks =  4

page =  6
Length of the layout 100
text blocks =  95
sep blocks =  1
figure blocks =  1

page =  7
Length of the layout 100
text blocks =  64
sep blocks =  12
figure blocks =  9

page =  8
Length of the layout 72
text blocks =  54
sep blocks =  7
figure blocks =  7

page =  9
Length of the layout 79
text blocks =  29
sep blocks =  3
figure blocks =  20

page =  10
Length of the layout 89
text blocks =  9
sep blocks =  6
figure blocks =  23

page =  11
Length of the layout 87
text blocks =  55
sep blocks =  8
figure blocks =  16

page =  12
Length of

In [10]:
df

,Page,Block,Text,Image path
0,1,1,BECHU GAUD involved if the expecting Gore Kami...,outputs/1/1.png
1,1,2,_ _ In parhament.\nneturn seized assets In 3 M...,outputs/1/2.png
2,1,3,PM to head army integration panel\ny g [ ae ol...,outputs/1/3.png
3,1,4,7 fs Pha ee dig .\n66 | have directed ms Peo B...,outputs/1/4.png
4,1,5,"POST REPORT\nKATHMANDU, JAN. 7\nPrime Minister...",outputs/1/5.png
...,...,...,...,...
712,12,39,"cattle, Nach,OSA, 9 ""Quy, ""ae\nSTANDS a AUTHOR...",outputs/12/39.png
713,12,40,Vocal Performance in a Group” Lakpa elaborates...,outputs/12/40.png
714,12,41,"Meanwnhue, band members are busy\nnning lyrics...",outputs/12/41.png
715,12,42,"The Kathmandu Post, Thursday, January 8, 2009\...",outputs/12/42.png


In [11]:
df.to_csv("final.csv")